# Clase 10 — Analizando el catálogo de Netflix

En este ejercicio asumirás el rol de un **Data Analyst** dentro de Netflix. El equipo de negocio quiere entender mejor cómo está distribuido el catálogo de la plataforma para tomar decisiones sobre futuras inversiones en contenido.

Para ello deberás utilizar las herramientas aprendidas en esta clase:

- `groupby()`
- funciones de agregación
- `agg()`
- `pivot_table()`
- `unstack()`
- gráficos con Pandas y Matplotlib

> No utilices bucles (`for`) para resolver los ejercicios.


## Parte 1: Exploración del dataset

Antes de comenzar el análisis responde:

1. Carga el dataset.
2. Muestra las primeras filas.
3. ¿Cuántas filas y columnas posee?
4. ¿Qué tipo de dato tiene cada columna?
5. ¿Existen valores nulos?
6. ¿Qué columnas presentan mayor cantidad de valores faltantes?

In [2]:
pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
# tú código acá
import pandas as pd

df = pd.read_csv("/workspaces/data-analysis-course/modulo-3-transformacion/clase-10/ventas_retail.csv")

df.groupby("ciudad")["monto"].sum()

ciudad
Buenos Aires    8855199
Córdoba         7198111
Mendoza         7706414
Rosario         7938539
Tucumán         7221605
Name: monto, dtype: int64

In [10]:
df.groupby(["ciudad", "categoria"])["monto"].sum()

ciudad        categoria  
Buenos Aires  Alimentos      1876997
              Deportes       1372041
              Electrónica    1951331
              Hogar          1607118
              Ropa           2047712
Córdoba       Alimentos      1787767
              Deportes       1778075
              Electrónica    1286509
              Hogar          1292147
              Ropa           1053613
Mendoza       Alimentos       891215
              Deportes       1718730
              Electrónica    1472067
              Hogar          1533021
              Ropa           2091381
Rosario       Alimentos      1715244
              Deportes       1676528
              Electrónica    1526021
              Hogar          1783981
              Ropa           1236765
Tucumán       Alimentos      2595971
              Deportes       1047502
              Electrónica    1445364
              Hogar          1324889
              Ropa            807879
Name: monto, dtype: int64

In [11]:
df.groupby("vendedor")["monto"].agg(["sum", "mean", "count", "max"])

,sum,mean,count,max
vendedor,,,,
Ana Torres,5592010,79885.857143,70,149832
Carlos Vera,5114603,75214.750000,68,145272
Clara Díaz,5051539,77715.984615,65,148273
Lucía Paz,4423650,77607.894737,57,145180
Luis Gómez,5188342,76299.147059,68,149701
María López,4380650,70655.645161,62,146398
Pedro Ruiz,4105135,78944.903846,52,146706
Sofía Herrera,5063939,87309.293103,58,149539


In [12]:
df.groupby("ciudad").agg(
    total_ventas   = ("monto",    "sum"),
    ticket_promedio= ("monto",    "mean"),
    unidades_total = ("unidades", "sum"),
    descuento_max  = ("descuento","max"),
)

,total_ventas,ticket_promedio,unidades_total,descuento_max
ciudad,,,,
Buenos Aires,8855199,81240.357798,1089,0.2
Córdoba,7198111,75769.589474,931,0.2
Mendoza,7706414,68807.267857,1175,0.2
Rosario,7938539,87236.692308,946,0.2
Tucumán,7221605,77651.666667,880,0.2


In [13]:
resultado = df.groupby("ciudad")["monto"].sum()
print(type(resultado.index))   # Index — el índice es la ciudad

resultado_plano = resultado.reset_index()
print(resultado_plano)

<class 'pandas.Index'>
         ciudad    monto
0  Buenos Aires  8855199
1       Córdoba  7198111
2       Mendoza  7706414
3       Rosario  7938539
4       Tucumán  7221605


In [14]:
tabla = df.pivot_table(
    values  = "monto",
    index   = "ciudad",
    columns = "categoria",
    aggfunc = "sum",
)

print(tabla)

categoria     Alimentos  Deportes  Electrónica    Hogar     Ropa
ciudad                                                          
Buenos Aires    1876997   1372041      1951331  1607118  2047712
Córdoba         1787767   1778075      1286509  1292147  1053613
Mendoza          891215   1718730      1472067  1533021  2091381
Rosario         1715244   1676528      1526021  1783981  1236765
Tucumán         2595971   1047502      1445364  1324889   807879


In [16]:
tabla = df.pivot_table(
    values   = "monto",
    index    = "ciudad",
    columns  = "categoria",
    aggfunc  = "sum",
    fill_value = 0,
)

print(tabla)

categoria     Alimentos  Deportes  Electrónica    Hogar     Ropa
ciudad                                                          
Buenos Aires    1876997   1372041      1951331  1607118  2047712
Córdoba         1787767   1778075      1286509  1292147  1053613
Mendoza          891215   1718730      1472067  1533021  2091381
Rosario         1715244   1676528      1526021  1783981  1236765
Tucumán         2595971   1047502      1445364  1324889   807879


In [17]:
# groupby con dos columnas → MultiIndex
por_ciudad_trim = df.groupby(["ciudad", "trimestre"])["monto"].sum()
print(por_ciudad_trim)

ciudad        trimestre
Buenos Aires  Q1           2556390
              Q2           2735824
              Q3           1607655
              Q4           1955330
Córdoba       Q1           1632413
              Q2           1596225
              Q3           1932245
              Q4           2037228
Mendoza       Q1           2500488
              Q2           1617051
              Q3           1907793
              Q4           1681082
Rosario       Q1           1957784
              Q2           2212611
              Q3           2461971
              Q4           1306173
Tucumán       Q1           1536296
              Q2           2079097
              Q3           1103564
              Q4           2502648
Name: monto, dtype: int64


In [18]:
# unstack() convierte el trimestre (nivel interno) en columnas
tabla_unstacked = por_ciudad_trim.unstack()
print(tabla_unstacked)

trimestre          Q1       Q2       Q3       Q4
ciudad                                          
Buenos Aires  2556390  2735824  1607655  1955330
Córdoba       1632413  1596225  1932245  2037228
Mendoza       2500488  1617051  1907793  1681082
Rosario       1957784  2212611  2461971  1306173
Tucumán       1536296  2079097  1103564  2502648


## Parte 2: Agrupaciones

### Ejercicio 1 - ¿Cuántos títulos existen por país?

Ordena el resultado de mayor a menor.


In [ ]:
# tu código acá

### Ejercicio 2 - ¿Cuántas películas y cuántas series tiene Netflix?

In [ ]:
# tu código acá

### Ejercicio 3 - ¿Cuántos títulos fueron lanzados cada año?

Ordena el resultado cronológicamente.


In [ ]:
# tu código acá

### Ejercicio 4 - ¿Cuál es la clasificación por edades más frecuente?

Por ejemplo:

- TV-MA
- TV-14
- PG
- R

In [ ]:
# tu código acá

### Ejercicio 5 - Obtén para cada tipo de contenido (Movie y TV Show):

- cantidad de títulos
- año promedio de lanzamiento
- año más reciente
- año más antiguo

> Utiliza **agg()**.

In [ ]:
# tu código acá

## Parte 3: Agrupaciones múltiples

### Ejercicio 6 - Obtén la cantidad de títulos para cada combinación.

Agrupa por:

- país
- tipo de contenido

In [ ]:
# tu código acá

### Ejercicio 7 - Convierte el resultado anterior en una tabla donde:

- las filas sean los países
- las columnas sean Movie y TV Show

> Utiliza **unstack()**.

In [ ]:
# tu código acá

## Parte 4: Tablas dinámicas

### Ejercicio 8 - Construye una tabla dinámica donde:

- filas → país
- columnas → tipo de contenido
- valores → cantidad de títulos

In [ ]:
# tu código acá

### Ejercicio 9 - Construye otra tabla dinámica donde:

- filas → clasificación por edades
- columnas → tipo de contenido
- valores → cantidad de títulos

In [ ]:
# tu código acá

### Ejercicio 10 - Modifica la tabla dinámica para que los valores faltantes aparezcan como **0**.

In [ ]:
# tu código acá

## Parte 5: Visualización

### Ejercicio 11 - Genera un gráfico de barras con el **Top 10 países con mayor cantidad de títulos**.

El gráfico debe incluir:

- título
- tamaño de figura
- etiquetas rotadas
- eje Y con nombre


In [ ]:
# tu código acá

### Ejercicio 12 - Genera un gráfico de barras mostrando la cantidad de:

- Movies
- TV Shows

In [ ]:
# tu código acá

### Ejercicio 13 - Utilizando la tabla dinámica, crea un gráfico de barras agrupadas que compare películas y series por país.


In [ ]:
# tu código acá

## Parte 6: Análisis

Redacta al menos **cinco insights** basados en el análisis realizado. Algunas preguntas que puedes responder son:

- ¿Qué país domina el catálogo?
- ¿Netflix posee más películas o más series?
- ¿Qué clasificación por edades es la más utilizada?
- ¿Existen países donde predominen las series?
- ¿Qué tendencia observas respecto al año de lanzamiento de los contenidos?

Justifica cada respuesta utilizando los datos obtenidos.

In [ ]:
# tu código acá

## Desafío

Netflix desea aumentar su presencia en Latinoamérica. Analiza únicamente los siguientes países:

- Argentina
- Brasil
- Chile
- Colombia
- México
- Perú

Responde:

1. ¿Cuál aporta mayor cantidad de contenido?
2. ¿Predominan las películas o las series?
3. ¿Qué clasificación por edades es la más común?
4. Genera un gráfico comparativo.
5. Escribe una breve recomendación para el director regional de Netflix basada en tus resultados.


In [ ]:
# tu código acá